In [3]:
from tavily import TavilyClient
from datetime import datetime, timedelta
import json
import asyncio
import json
from typing import List, Dict, Any, Optional

from pydantic import BaseModel, Field, ValidationError
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Import shared clients
import shared_clients
from shared_clients import shared_clients


#!/usr/bin/env python3
"""
Manager Multiprocessing - Import Manager Agent and run multiprocessing analysis
"""

import asyncio
import multiprocessing as mp
from typing import List, Dict, Any
from concurrent.futures import ProcessPoolExecutor
from time import time


#!/usr/bin/env python3
"""
Simple Multiprocessing Manager - Direct multiprocessing without classes
"""

import asyncio
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from Manager_Agent import quick_analysis

import asyncio
import json
from typing import Dict, Any

# Import shared clients
import shared_clients
from shared_clients import shared_clients

ModuleNotFoundError: No module named 'shared_clients'

In [2]:
# To install: pip install tavily-python
from tavily import TavilyClient

def get_company_name_by_ticker(ticker: str) -> str:
    """
    Get company name from ticker symbol using Tavily search.
    
    Args:
        ticker (str): Stock ticker symbol (e.g., "BBW", "AAPL", "TSLA")
    
    Returns:
        str: Company name
    """
    client = TavilyClient("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")
    
    # Enhanced query to get company name only
    query = f"what is the company name for stock ticker {ticker.upper()}, return only the company name"
    
    try:
        response = client.search(
            query=query,
            include_answer="advanced",
            search_depth="advanced"
        )
        
        # Extract company name from response
        if response and 'answer' in response:
            company_name = response['answer']
            print(f"✅ Found company name for {ticker}: {company_name}")
            return company_name
        else:
            print(f"❌ No company name found for ticker {ticker}")
            return f"Unknown Company ({ticker})"
            
    except Exception as e:
        print(f"❌ Error searching for {ticker}: {e}")
        return f"Error ({ticker})"

def get_stock_analysis_tavily_dual(company_name):
    """
    Get stock analysis using two Tavily searches and integrate results
    
    Args:
        company_name (str): Company name (e.g., 'Build-A-Bear Workshop', 'Apple Inc.')
    
    Returns:
        dict: Contains summary['sell_and_buy'] and summary['business_logic']
    """
    try:
        # Initialize Tavily client
        client = TavilyClient("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")
        
        print(f"🔍 Starting dual Tavily search for {company_name}...")
        
        # SEARCH 1: Sell/Buy Analysis (Market focus, validation, fundamental level)
        query1 = f"List 3–9 stock drivers (Macro/Market/Fundamentals) for {company_name}, with +/– factors tied to future growth, risks, or resolutions. Return ONLY numbered list(each line summarize as key words bullet point)."
        print(f"📊 Search 1: Sell/Buy analysis for {company_name}...")
        
        response1 = client.search(
            query=query1,
            include_answer="advanced",
            search_depth="advanced",
            topic="general",
            max_results=10
        )
        
        # SEARCH 2: Business Logic Analysis (Moat, business model)
        query2 = f"I want a very specific deep detailed explanation on the business model: moat, scale, customers, and logic for {company_name}, if no, just say some positive and negative about the company"
        
        print(f"�� Search 2: Business logic analysis for {company_name}...")
        
        response2 = client.search(
            query=query2,
            include_answer="advanced",
            search_depth="advanced",
            topic="general",
            max_results=10
        )
        
        # Extract content and answer from Search 1 (Sell/Buy)
        sell_buy_content_list = []
        if 'results' in response1:
            for result in response1['results']:
                if 'content' in result and result['content']:
                    sell_buy_content_list.append(result['content'])
        
        sell_buy_content = " | ".join(sell_buy_content_list)
        sell_buy_answer = response1.get('answer', '')
        
        # Extract content and answer from Search 2 (Business Logic)
        business_logic_content_list = []
        if 'results' in response2:
            for result in response2['results']:
                if 'content' in result and result['content']:
                    business_logic_content_list.append(result['content'])
        
        business_logic_content = " | ".join(business_logic_content_list)
        business_logic_answer = response2.get('answer', '')
        
        # Create structured summary
        summary = {
            'sell_and_buy': sell_buy_answer,  # This will now be a numbered list
            'business_logic': business_logic_answer
        }
        
        # Create result dictionary
        result = {
            'company_name': company_name,
            'summary': summary,
            'sell_and_buy_content': sell_buy_content,
            'business_logic_content': business_logic_content,
            'sell_and_buy_response_time': response1.get('response_time', 0),
            'business_logic_response_time': response2.get('response_time', 0),
            'sell_and_buy_results': len(response1.get('results', [])),
            'business_logic_results': len(response2.get('results', []))
        }
        
        print(f"✅ Dual Tavily search complete for {company_name}")
        print(f"💰 Sell/Buy results: {result['sell_and_buy_results']}")
        print(f"🏢 Business Logic results: {result['business_logic_results']}")
        print(f"⏱️ Total response time: {result['sell_and_buy_response_time'] + result['business_logic_response_time']} seconds")
        
        return result
        
    except Exception as e:
        print(f"❌ Error in dual Tavily search for {company_name}: {e}")
        return {
            'company_name': company_name,
            'summary': {
                'sell_and_buy': 'Error occurred during sell/buy search',
                'business_logic': 'Error occurred during business logic search'
            },
            'sell_and_buy_content': '',
            'business_logic_content': ''
        }

# MAIN PIPELINE FUNCTION
def get_stock_analysis_pipeline(ticker):
    """
    Complete pipeline: Ticker -> Company Name -> Two Searches
    
    Args:
        ticker (str): Stock ticker symbol (e.g., "BBW")
    
    Returns:
        dict: Complete analysis with company name and two search results
    """
    print(f"🚀 Starting analysis pipeline for ticker: {ticker}")
    
    # STEP 1: Get company name from ticker
    company_name = get_company_name_by_ticker(ticker)
    
    # STEP 2: Use company name for two searches
    analysis = get_stock_analysis_tavily_dual(company_name)
    
    # Add ticker to result
    analysis['ticker'] = ticker
    
    return analysis

# Simple usage function
def get_stock_summary_simple(ticker):
    """
    Simple function to get stock summary - just input ticker
    
    Args:
        ticker (str): Stock ticker symbol
    
    Returns:
        dict: Summary with sell_and_buy and business_logic
    """
    result = get_stock_analysis_pipeline(ticker)
    return result['summary']


In [3]:
ticker = "MSFT"
language = "Chinese"
company_name = get_company_name_by_ticker(ticker)
print(f"Company: {company_name}")

# Test stock analysis
analysis = get_stock_summary_simple(ticker)
print(f"Sell/Buy Analysis: {analysis['sell_and_buy']}")
print(f"Business Logic: {analysis['business_logic']}")
sell_and_buy = analysis['sell_and_buy']
business_logic = analysis['business_logic']

✅ Found company name for MSFT: Microsoft Corporation
Company: Microsoft Corporation
🚀 Starting analysis pipeline for ticker: MSFT
✅ Found company name for MSFT: Microsoft Corporation
🔍 Starting dual Tavily search for Microsoft Corporation...
📊 Search 1: Sell/Buy analysis for Microsoft Corporation...
�� Search 2: Business logic analysis for Microsoft Corporation...


KeyboardInterrupt: 